# Colab Session A — QLoRA Retrain + KB v2 Rebuild

**Runtime:** A100 High-RAM (change runtime before running!)
**Est. time:** ~2h wall clock
**Tasks:**
1. QLoRA retrain — 500 steps, batch_size=8, lr=2e-4
2. Knowledge base rebuild with `RecursiveSentenceChunker` (KB v2)
3. Push artifacts to Google Drive

**Prerequisites:**
- Upload KB v1 + models to Google Drive before starting
- `GITHUB_REPO` below must point to your repo
- Run Session B notebook **in parallel** to save wall-clock time


In [ ]:
# ── CONFIGURATION ────────────────────────────────────────────────────────────
GITHUB_REPO   = "https://github.com/kbssrikar7/final_project.git"
GITHUB_BRANCH = "main"
DRIVE_BASE    = "/content/drive/MyDrive/healthcare_qa"

# Paths inside Colab
PROJECT_DIR   = "/content/project"
KB_V1_DIR     = f"{PROJECT_DIR}/data/knowledge_base"
KB_V2_DIR     = f"{PROJECT_DIR}/data/knowledge_base_v2"
MODELS_DIR    = f"{PROJECT_DIR}/models"

print("Configuration loaded")

In [ ]:
# ── A1: Environment Setup ─────────────────────────────────────────────────────
import subprocess, os

# Verify GPU
import torch
assert torch.cuda.is_available(), "No GPU detected! Switch runtime to A100 High-RAM"
print(f"GPU: {torch.cuda.get_device_name(0)}  |  VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

# Clone repo
if not os.path.exists(PROJECT_DIR):
    subprocess.run(["git", "clone", "--branch", GITHUB_BRANCH, GITHUB_REPO, PROJECT_DIR], check=True)
else:
    subprocess.run(["git", "-C", PROJECT_DIR, "pull", "origin", GITHUB_BRANCH], check=True)
os.chdir(PROJECT_DIR)
print("Repo ready")

# Install deps
subprocess.run([
    "pip", "install", "-q", "-r", "requirements.txt",
    "--extra-index-url", "https://download.pytorch.org/whl/cu118"
], check=True)
print("Dependencies installed")

In [ ]:
# ── A2: Mount Drive + Sync Data ──────────────────────────────────────────────
from google.colab import drive
drive.mount("/content/drive")

import shutil, os

# Sync KB v1 from Drive (needed only for reference; v2 will be built fresh)
drive_kb = f"{DRIVE_BASE}/knowledge_base"
if os.path.exists(drive_kb):
    print(f"Syncing KB v1 from Drive ({drive_kb})...")
    shutil.copytree(drive_kb, KB_V1_DIR, dirs_exist_ok=True)
    print("KB v1 synced")
else:
    print("WARNING: KB v1 not found on Drive — raw data rebuild will take longer")

# Sync models (base model + existing adapter)
drive_models = f"{DRIVE_BASE}/models"
if os.path.exists(drive_models):
    shutil.copytree(drive_models, MODELS_DIR, dirs_exist_ok=True)
    print("Models synced from Drive")
else:
    print("INFO: No pre-uploaded models on Drive — will download from HuggingFace")

# Sync raw data for KB rebuild
drive_raw = f"{DRIVE_BASE}/data_raw"
raw_dir = f"{PROJECT_DIR}/data/raw"
if os.path.exists(drive_raw):
    shutil.copytree(drive_raw, raw_dir, dirs_exist_ok=True)
    print("Raw data synced from Drive")

In [ ]:
# ── A3: QLoRA Retrain — 500 steps ────────────────────────────────────────────
# Expected time: ~15 min on A100 with bs=8
import subprocess, time

t0 = time.time()
result = subprocess.run([
    "python3", "src/fine_tuning/trainer.py",
    "--steps", "500",
    "--lr", "2e-4",
    "--batch-size", "8",
    "--output-dir", "models/fine_tuned/medical_adapter_v2",
], capture_output=False)
elapsed = time.time() - t0

if result.returncode == 0:
    print(f"QLoRA retrain complete in {elapsed/60:.1f} min")
    import os
    adapter_path = f"{PROJECT_DIR}/models/fine_tuned/medical_adapter_v2"
    if os.path.exists(adapter_path):
        files = os.listdir(adapter_path)
        print(f"Adapter files: {files}")
    else:
        print("WARNING: Adapter output directory not found")
else:
    print(f"ERROR: QLoRA retrain failed (rc={result.returncode}). Check output above.")

In [ ]:
# ── A4: KB v2 Rebuild with RecursiveSentenceChunker ──────────────────────────
# Expected time: ~60 min on A100 for ~400K chunks
import subprocess, time

print("Building KB v2 with RecursiveSentenceChunker...")
print("This uses domain-adaptive sizes: PubMedQA=256tok, MedMCQA=128tok, HCM=512tok")

t0 = time.time()
result = subprocess.run([
    "python3", "scripts/build_knowledge_base.py",
    "--chunker", "recursive",
    "--output-dir", "data/knowledge_base_v2",
    "--collection", "medical_knowledge_v2",
], capture_output=False)
elapsed = time.time() - t0

if result.returncode == 0:
    print(f"KB v2 build complete in {elapsed/60:.1f} min")
else:
    print(f"WARNING: KB build returned rc={result.returncode}. Check output above.")

In [ ]:
# ── A5: Verify KB v2 + Write embedding_meta.json ─────────────────────────────
import os, json
from pathlib import Path

kb_v2_path = Path(f"{PROJECT_DIR}/data/knowledge_base_v2")
meta_path  = kb_v2_path / "embedding_metadata.json"

if kb_v2_path.exists():
    print(f"KB v2 directory: {kb_v2_path}")
    print(f"Files: {list(kb_v2_path.iterdir())[:10]}")

    if meta_path.exists():
        meta = json.loads(meta_path.read_text())
        print(f"Embedding meta: {meta}")
    else:
        print("WARNING: embedding_metadata.json not written — check build script")

    # Quick sanity: load vector store and check doc count
    import sys; sys.path.insert(0, PROJECT_DIR)
    import chromadb
    client = chromadb.PersistentClient(path=str(kb_v2_path))
    try:
        coll = client.get_collection("medical_knowledge_v2")
        count = coll.count()
        print(f"KB v2 document count: {count:,}")
        if count < 100_000:
            print("WARNING: count seems low — expected ~150K-220K")
        else:
            print("OK: count looks reasonable")
    except Exception as e:
        print(f"Could not query KB v2: {e}")
else:
    print(f"ERROR: KB v2 directory not found at {kb_v2_path}")

In [ ]:
# ── A6: Push Artifacts to Drive ──────────────────────────────────────────────
import shutil, os
from pathlib import Path

os.makedirs(DRIVE_BASE, exist_ok=True)

# Push adapter v2
adapter_src = Path(f"{PROJECT_DIR}/models/fine_tuned/medical_adapter_v2")
adapter_dst = Path(f"{DRIVE_BASE}/models/fine_tuned/medical_adapter_v2")
if adapter_src.exists():
    shutil.copytree(adapter_src, adapter_dst, dirs_exist_ok=True)
    print(f"Adapter v2 pushed to Drive: {adapter_dst}")
else:
    print("WARNING: Adapter v2 not found — skipping Drive push")

# Push KB v2
kb_v2_src = Path(f"{PROJECT_DIR}/data/knowledge_base_v2")
kb_v2_dst = Path(f"{DRIVE_BASE}/knowledge_base_v2")
if kb_v2_src.exists():
    print("Pushing KB v2 to Drive (~2-3 GB, may take a few minutes)...")
    shutil.copytree(kb_v2_src, kb_v2_dst, dirs_exist_ok=True)
    print(f"KB v2 pushed to Drive: {kb_v2_dst}")
else:
    print("ERROR: KB v2 not found — nothing to push")

print("Session A complete. Start Session C once Session B finishes.")